In [1]:
import warnings, pathlib
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

IMG   = pathlib.Path('images')
DATOS = pathlib.Path('datos')
IMG.mkdir(exist_ok=True)
DATOS.mkdir(exist_ok=True)

print("Entorno listo.")
print(f"  numpy  {np.__version__}")
print(f"  pandas {pd.__version__}")
print(f"  sklearn {__import__('sklearn').__version__}")


Entorno listo.
  numpy  1.26.4
  pandas 2.2.3
  sklearn 1.4.1.post1


# PCA y Reducción de Dimensionalidad

## 1 · Contexto y objetivo de aprendizaje

Cuando un modelo de ML recibe muchas variables de entrada, necesita más datos
para detectar patrones reales y no aprender ruido. Este fenómeno - la
**maldición de la dimensionalidad** - aparece en la práctica cuando el número
de columnas del dataset crece sin que crezca el número de filas.

La respuesta técnica es la **reducción de dimensionalidad**: comprimir las
variables originales en un número menor de variables sintéticas que conservan
la mayor parte de la información.

La técnica más usada es el **Análisis de Componentes Principales (PCA,
Principal Component Analysis)**.

**Al terminar este notebook sabrás:**

- por qué más variables no siempre es mejor
- qué hace PCA geométricamente
- cómo elegir cuántos componentes conservar
- cuándo PCA ayuda y cuándo perjudica

---

> **Antes de seguir:** ¿Qué haría yo si quisiera añadir 20 variables nuevas a
> un modelo que ya tengo entrenado con 300 registros? ¿Qué pregunta debería
> hacerme antes de añadirlas?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

Una respuesta madura menciona: que añadir variables sin añadir datos puede
empeorar el modelo; que hay que evaluar si los datos son suficientes para
las variables que se quieren usar; que algunas variables pueden estar
correlacionadas y aportar información redundante.

Si nadie responde, preguntar: "Si tenemos 300 clientes y 20 variables nuevas,
¿cuántos ejemplos tenemos por variable?"

Señal de comprensión: el alumno distingue entre relevancia de una variable
(¿aporta información?) y viabilidad estadística (¿tengo suficientes datos
para aprender lo que aporta?).

</details>


## 2 · Dataset: métricas de uso de ERP (sintético)

In [2]:
# Generamos 200 clientes con 10 métricas de uso del ERP
# Las variables tienen correlaciones realistas entre sí

n = 200

# Perfil latente: intensidad de uso (0-1) y nivel de fricción (0-1)
uso       = np.random.beta(2, 2, n)          # intensidad general de uso
friccion  = np.random.beta(1.5, 3, n)        # señales de problemas

# Variables correlacionadas con el perfil de uso
login_frecuencia        = np.clip(uso * 30  + np.random.normal(0, 3,  n), 1, 35).round(1)
modulos_activos         = np.clip(uso * 8   + np.random.normal(0, 1,  n), 1, 10).round().astype(int)
sesiones_semana         = np.clip(uso * 15  + np.random.normal(0, 2,  n), 1, 20).round(1)
volumen_transacciones   = np.clip(uso * 500 + np.random.normal(0, 60, n), 10, 600).round()

# Variables correlacionadas con fricción
tickets_mes             = np.clip(friccion * 8 + np.random.normal(0, 1, n), 0, 12).round().astype(int)
incidencias_criticas    = np.clip(friccion * 3 + np.random.normal(0, 0.5, n), 0, 5).round().astype(int)
dias_sin_login          = np.clip((1 - uso) * 60 + np.random.normal(0, 5, n), 0, 90).round().astype(int)

# Variables mixtas
nps                     = np.clip((uso - friccion) * 100 + np.random.normal(0, 10, n), -100, 100).round().astype(int)
antiguedad_meses        = np.clip(np.random.exponential(24, n), 1, 84).round().astype(int)
tam_empresa_empleados   = np.random.choice([10, 25, 50, 100, 250, 500], n,
                                            p=[0.20, 0.25, 0.25, 0.15, 0.10, 0.05])

# Segmento (variable de grupo para visualización, no usada en PCA)
score = uso - friccion * 0.5
segmento = pd.cut(score, bins=[-np.inf, 0.25, 0.55, np.inf],
                  labels=['bajo', 'moderado', 'intensivo'])

df = pd.DataFrame({
    'login_frecuencia':      login_frecuencia,
    'modulos_activos':       modulos_activos,
    'sesiones_semana':       sesiones_semana,
    'volumen_transacciones': volumen_transacciones,
    'tickets_mes':           tickets_mes,
    'incidencias_criticas':  incidencias_criticas,
    'dias_sin_login':        dias_sin_login,
    'nps':                   nps,
    'antiguedad_meses':      antiguedad_meses,
    'tam_empresa_empleados': tam_empresa_empleados,
    'segmento':              segmento,
})

df.to_csv(DATOS / 'metricas_uso_erp.csv', index=False)
print(f"Dataset guardado: {DATOS / 'metricas_uso_erp.csv'}")
print(f"Shape: {df.shape}")
print(f"\nPrimeras filas:")
df.head()


Dataset guardado: datos\metricas_uso_erp.csv
Shape: (200, 11)

Primeras filas:


,login_frecuencia,modulos_activos,sesiones_semana,volumen_transacciones,tickets_mes,incidencias_criticas,dias_sin_login,nps,antiguedad_meses,tam_empresa_empleados,segmento
0,21.3,4,7.9,309.0,3,2,17,10,20,50,moderado
1,19.4,5,4.7,151.0,0,1,25,16,30,250,moderado
2,24.0,4,5.6,381.0,0,0,28,66,2,25,intensivo
3,11.7,1,6.4,106.0,0,0,38,22,13,250,moderado
4,28.9,7,13.2,471.0,1,0,1,78,3,50,intensivo


In [3]:
# Sanity checks
assert df.shape == (200, 11), f"Shape inesperado: {df.shape}"
assert df.isnull().sum().sum() == 0, "Hay NaNs en el dataset"
print("Sanity checks: OK")
print(f"\nDistribución de segmentos:")
print(df['segmento'].value_counts().to_string())


Sanity checks: OK

Distribución de segmentos:
segmento
moderado     81
bajo         74
intensivo    45


## 3 · El problema: visualizar 10 variables simultáneamente es inmanejable

Con 10 variables necesitaríamos un espacio de 10 dimensiones para representar
cada cliente como un punto. No podemos dibujarlo.

La alternativa habitual - una matriz de scatter plots entre pares de variables - 
produce C(10,2) = 45 gráficos distintos. Miramos la versión reducida (4×4)
para ver el problema de escala.


In [4]:
FEATURES = [
    'login_frecuencia', 'modulos_activos', 'sesiones_semana',
    'volumen_transacciones', 'tickets_mes', 'incidencias_criticas',
    'dias_sin_login', 'nps'
]

color_map = {'intensivo': '#2196F3', 'moderado': '#FF9800', 'bajo': '#F44336'}
colors    = df['segmento'].map(color_map)

fig, axes = plt.subplots(4, 4, figsize=(12, 10))
fig.suptitle('Scatter matrix - 8 variables (4×4 = 16 gráficos)\n'
             'Con 10 variables serían 45 gráficos', fontsize=12, y=1.01)

for i, row_feat in enumerate(FEATURES[:4]):
    for j, col_feat in enumerate(FEATURES[:4]):
        ax = axes[i][j]
        if i == j:
            ax.hist(df[row_feat], bins=20, color='#90CAF9', edgecolor='white')
        else:
            ax.scatter(df[col_feat], df[row_feat], c=colors, alpha=0.5, s=15)
        if i == 3:
            ax.set_xlabel(col_feat, fontsize=7)
        if j == 0:
            ax.set_ylabel(row_feat, fontsize=7)
        ax.tick_params(labelsize=6)

patches = [mpatches.Patch(color=c, label=s) for s, c in color_map.items()]
fig.legend(handles=patches, loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(IMG / 'B02D_fig00_scatter_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print("Guardado: B02D_fig00_scatter_matrix.png")
print("\n→ 4x4 ya es difícil de leer. Con 10 variables: 45 gráficos.")
print("→ PCA comprime toda esta información en 2 ejes interpretables.")


Guardado: B02D_fig00_scatter_matrix.png

→ 4x4 ya es difícil de leer. Con 10 variables: 45 gráficos.
→ PCA comprime toda esta información en 2 ejes interpretables.


## 4 · PCA paso a paso: qué hace internamente (sin librería)

Antes de usar sklearn, veamos los 3 pasos que PCA ejecuta internamente.
El objetivo no es implementarlo: es entender **qué decisión geométrica toma**.

### Paso 1: estandarizar las variables

PCA trabaja con varianzas. Si una variable está en miles (volumen de
transacciones) y otra en unidades (módulos activos), la primera dominaría
el análisis simplemente por su escala, no por su relevancia.

La solución es estandarizar: transformar cada variable para que tenga
media 0 y desviación estándar 1.


In [5]:
X = df[FEATURES].values.astype(float)

# Estandarización manual
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)
X_std_arr = (X - X_mean) / X_std

print("Antes de estandarizar - medias y desviaciones:")
for i, f in enumerate(FEATURES):
    print(f"  {f:<28} media={X[:,i].mean():8.2f}  std={X[:,i].std():7.2f}")

print("\nDespués de estandarizar - medias y desviaciones:")
for i, f in enumerate(FEATURES):
    print(f"  {f:<28} media={X_std_arr[:,i].mean():8.4f}  std={X_std_arr[:,i].std():.4f}")


Antes de estandarizar -  medias y desviaciones:
  login_frecuencia             media=   15.42  std=   6.97
  modulos_activos              media=    4.13  std=   1.99
  sesiones_semana              media=    7.63  std=   3.64
  volumen_transacciones        media=  249.75  std= 120.37
  tickets_mes                  media=    2.65  std=   1.79
  incidencias_criticas         media=    1.05  std=   0.77
  dias_sin_login               media=   29.55  std=  14.35
  nps                          media=   17.07  std=  31.49

Después de estandarizar -  medias y desviaciones:
  login_frecuencia             media=  0.0000  std=1.0000
  modulos_activos              media=  0.0000  std=1.0000
  sesiones_semana              media= -0.0000  std=1.0000
  volumen_transacciones        media=  0.0000  std=1.0000
  tickets_mes                  media=  0.0000  std=1.0000
  incidencias_criticas         media= -0.0000  std=1.0000
  dias_sin_login               media= -0.0000  std=1.0000
  nps                  

### Paso 2: calcular la matriz de covarianza

La matriz de covarianza mide cuánto varían juntas cada par de variables.
Si dos variables tienen covarianza alta, contienen información similar
(están correlacionadas). PCA detecta y combina esas redundancias.


In [6]:
# Covarianza: C[i,j] = cómo varían juntas las variables i y j
C = np.cov(X_std_arr, rowvar=False)   # shape (8, 8)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels([f.replace('_', '\n') for f in FEATURES], fontsize=7)
ax.set_yticklabels(FEATURES, fontsize=7)
plt.colorbar(im, ax=ax, label='Covarianza')
ax.set_title('Matriz de covarianza estandarizada\n'
             '(valores cercanos a ±1 = alta correlación)', fontsize=10)
plt.tight_layout()
plt.savefig(IMG / 'B02D_fig00b_cov_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print("Guardado: B02D_fig00b_cov_matrix.png")
print("\nVariables con alta correlación positiva comparten información.")
print("PCA las combinará para evitar contarlas dos veces.")


Guardado: B02D_fig00b_cov_matrix.png

Variables con alta correlación positiva comparten información.
PCA las combinará para evitar contarlas dos veces.


### Paso 3: encontrar los autovectores (ejes de máxima varianza)

Los **autovectores** de la matriz de covarianza son las direcciones en el
espacio de datos en las que los datos se "extienden" más. El primer
autovector apunta en la dirección de máxima varianza; el segundo, en la
segunda dirección de máxima varianza, perpendicular a la primera; y así.

Cada autovector se convierte en un **componente principal**: una nueva
variable sintética que es una combinación lineal de las originales.

El **autovalor** asociado a cada autovector indica cuánta varianza del
dataset captura ese componente. Dividiendo cada autovalor entre la suma de
todos, obtenemos el porcentaje de varianza explicada.


In [7]:
eigenvalues, eigenvectors = np.linalg.eigh(C)

# eigh devuelve orden ascendente → invertimos
eigenvalues  = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

varianza_explicada      = eigenvalues / eigenvalues.sum()
varianza_acumulada      = varianza_explicada.cumsum()

print(f"{'Componente':<12} {'Autovalor':>10} {'Varianza %':>12} {'Acumulada %':>13}")
print("-" * 52)
for i, (ev, vp, va) in enumerate(zip(eigenvalues, varianza_explicada, varianza_acumulada)):
    marca = " ← codo" if i == 2 else ""
    print(f"  PC{i+1:<9} {ev:>10.3f} {vp*100:>11.1f}% {va*100:>12.1f}%{marca}")


Componente    Autovalor   Varianza %   Acumulada %
----------------------------------------------------
  PC1              4.826        60.0%         60.0%
  PC2              1.794        22.3%         82.3%
  PC3              0.452         5.6%         88.0% ← codo
  PC4              0.278         3.5%         91.4%
  PC5              0.244         3.0%         94.4%
  PC6              0.217         2.7%         97.2%
  PC7              0.117         1.5%         98.6%
  PC8              0.112         1.4%        100.0%


## 5 · PCA con scikit-learn: la versión práctica

In [8]:
# Pipeline estándar: estandarizar primero, luego PCA
pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=8)),   # todos los componentes para analizar
])

X_pca_all = pipe_pca.fit_transform(df[FEATURES])
pca_model = pipe_pca.named_steps['pca']

varianza_sklearn    = pca_model.explained_variance_ratio_
varianza_acum_sk    = varianza_sklearn.cumsum()

print("Varianza explicada por componente (sklearn):")
print(f"{'PC':<5} {'Varianza %':>10} {'Acumulada %':>12}")
print("-" * 30)
for i, (v, va) in enumerate(zip(varianza_sklearn, varianza_acum_sk)):
    marca = " ✓ ≥85%" if va >= 0.85 and (i == 0 or varianza_acum_sk[i-1] < 0.85) else ""
    print(f"  PC{i+1:<3} {v*100:>9.1f}% {va*100:>11.1f}%{marca}")


Varianza explicada por componente (sklearn):
PC    Varianza %  Acumulada %
------------------------------
  PC1        60.0%        60.0%
  PC2        22.3%        82.3%
  PC3         5.6%        88.0% ✓ ≥85%
  PC4         3.5%        91.4%
  PC5         3.0%        94.4%
  PC6         2.7%        97.2%
  PC7         1.5%        98.6%
  PC8         1.4%       100.0%


## 6 · Scree plot: cuántos componentes conservar (B02D_fig01)

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

n_comp = len(varianza_sklearn)
comp_idx = range(1, n_comp + 1)

# Gráfico 1: varianza individual
ax1.bar(comp_idx, varianza_sklearn * 100, color='#42A5F5', edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Componente principal', fontsize=11)
ax1.set_ylabel('Varianza explicada (%)', fontsize=11)
ax1.set_title('Varianza por componente\n(scree plot)', fontsize=11)
ax1.set_xticks(comp_idx)
ax1.set_xticklabels([f'PC{i}' for i in comp_idx])

# Gráfico 2: varianza acumulada
ax2.plot(comp_idx, varianza_acum_sk * 100, 'o-', color='#1565C0', linewidth=2, markersize=8)
ax2.axhline(y=85, color='#E53935', linestyle='--', linewidth=1.5, label='Umbral 85%')

# Marcar el punto de corte
k_85 = int(np.argmax(varianza_acum_sk >= 0.85)) + 1
ax2.axvline(x=k_85, color='#E53935', linestyle=':', linewidth=1.5)
ax2.scatter([k_85], [varianza_acum_sk[k_85-1]*100], color='#E53935', s=120, zorder=5,
            label=f'k={k_85} → {varianza_acum_sk[k_85-1]*100:.1f}% varianza')
ax2.set_xlabel('Número de componentes (k)', fontsize=11)
ax2.set_ylabel('Varianza acumulada (%)', fontsize=11)
ax2.set_title('Varianza acumulada\n(criterio del codo)', fontsize=11)
ax2.set_xticks(comp_idx)
ax2.set_xticklabels([f'PC{i}' for i in comp_idx])
ax2.legend(fontsize=9)
ax2.set_ylim(0, 105)

plt.suptitle('PCA - Análisis de varianza explicada', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(IMG / 'B02D_fig01.png', dpi=120, bbox_inches='tight')
plt.close()
print(f"Guardado: B02D_fig01.png")
print(f"\n→ Con k={k_85} componentes se retiene ≥85% de la varianza original.")
print(f"   Pasamos de 8 variables a {k_85}, manteniendo el {varianza_acum_sk[k_85-1]*100:.1f}% de la información.")


Guardado: B02D_fig01.png

→ Con k=3 componentes se retiene ≥85% de la varianza original.
   Pasamos de 8 variables a 3, manteniendo el 88.0% de la información.


## 7 · Scatter 2D: los datos proyectados en los 2 primeros componentes (B02D_fig02)

In [10]:
# Proyectamos sobre los 2 primeros componentes
pipe_2d = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=2)),
])
X_2d = pipe_2d.fit_transform(df[FEATURES])

color_map = {'intensivo': '#2196F3', 'moderado': '#FF9800', 'bajo': '#F44336'}
fig, ax = plt.subplots(figsize=(8, 6))

for seg, color in color_map.items():
    mask = df['segmento'] == seg
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=color, label=seg, alpha=0.7, s=50, edgecolors='white', linewidth=0.4)

pca_2d = pipe_2d.named_steps['pca']
v1, v2 = pca_2d.explained_variance_ratio_
ax.set_xlabel(f'Componente 1  ({v1*100:.1f}% varianza)', fontsize=11)
ax.set_ylabel(f'Componente 2  ({v2*100:.1f}% varianza)', fontsize=11)
ax.set_title(f'Clientes en el espacio PCA-2D\n'
             f'Total varianza captada: {(v1+v2)*100:.1f}%', fontsize=12)
ax.legend(title='Segmento', fontsize=9)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.savefig(IMG / 'B02D_fig02.png', dpi=120, bbox_inches='tight')
plt.close()
print("Guardado: B02D_fig02.png")
print(f"\nEn 2 dimensiones (en lugar de 8) se puede ver la separación de segmentos.")
print(f"Varianza total captada: {(v1+v2)*100:.1f}%")


Guardado: B02D_fig02.png

En 2 dimensiones (en lugar de 8) se puede ver la separación de segmentos.
Varianza total captada: 82.3%


## 8 · Loadings: qué variable original contribuye a cada componente (B02D_fig03)

Los **loadings** son los pesos con los que cada variable original contribuye
a cada componente. Un loading alto (positivo o negativo) indica que esa
variable tiene mucha influencia en ese componente.

Este mapa permite interpretar qué significa cada componente en términos
de negocio.


In [11]:
pipe_k = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=k_85)),
])
pipe_k.fit(df[FEATURES])
pca_k = pipe_k.named_steps['pca']

loadings = pd.DataFrame(
    pca_k.components_.T,
    index=FEATURES,
    columns=[f'PC{i+1}' for i in range(k_85)]
)

fig, ax = plt.subplots(figsize=(k_85 * 1.4 + 2, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Loading (peso en el componente)'})
ax.set_title(f'Loadings - contribución de cada variable a los {k_85} componentes\n'
             f'(retienen {varianza_acum_sk[k_85-1]*100:.1f}% de la varianza total)',
             fontsize=11)
ax.set_xlabel('Componente principal', fontsize=10)
ax.set_ylabel('Variable original', fontsize=10)
plt.tight_layout()
plt.savefig(IMG / 'B02D_fig03.png', dpi=120, bbox_inches='tight')
plt.close()
print("Guardado: B02D_fig03.png")

# Interpretación automática de los 2 primeros componentes
for pc_idx in range(min(2, k_85)):
    top_pos = loadings[f'PC{pc_idx+1}'].nlargest(3)
    top_neg = loadings[f'PC{pc_idx+1}'].nsmallest(3)
    print(f"\nPC{pc_idx+1} - variables con mayor peso positivo: {list(top_pos.index)}")
    print(f"PC{pc_idx+1} - variables con mayor peso negativo: {list(top_neg.index)}")


Guardado: B02D_fig03.png

PC1 -  variables con mayor peso positivo: ['dias_sin_login', 'tickets_mes', 'incidencias_criticas']
PC1 -  variables con mayor peso negativo: ['login_frecuencia', 'volumen_transacciones', 'modulos_activos']

PC2 -  variables con mayor peso positivo: ['incidencias_criticas', 'tickets_mes', 'modulos_activos']
PC2 -  variables con mayor peso negativo: ['nps', 'dias_sin_login', 'volumen_transacciones']


## 9 · Varianza acumulada vs número de componentes (B02D_fig04)

In [12]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.fill_between(comp_idx, varianza_acum_sk * 100, alpha=0.15, color='#1565C0')
ax.plot(comp_idx, varianza_acum_sk * 100, 'o-', color='#1565C0',
        linewidth=2.5, markersize=9, label='Varianza acumulada')

umbrales = [(80, '#FFA726', '80%'), (85, '#E53935', '85%'), (95, '#7B1FA2', '95%')]
for val, col, lbl in umbrales:
    ax.axhline(y=val, color=col, linestyle='--', linewidth=1.2, alpha=0.8, label=f'Umbral {lbl}')

for i, va in enumerate(varianza_acum_sk):
    ax.annotate(f'{va*100:.0f}%', (i+1, va*100),
                textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8, color='#1565C0')

ax.set_xlabel('Número de componentes retenidos (k)', fontsize=11)
ax.set_ylabel('Varianza del dataset explicada (%)', fontsize=11)
ax.set_title('¿Cuántos componentes necesito?\n'
             'Varianza acumulada según k', fontsize=12)
ax.set_xticks(comp_idx)
ax.set_xticklabels([f'k={i}' for i in comp_idx])
ax.set_ylim(0, 110)
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(IMG / 'B02D_fig04.png', dpi=120, bbox_inches='tight')
plt.close()
print("Guardado: B02D_fig04.png")


Guardado: B02D_fig04.png


## 10 · Tabla comparativa: PCA vs selección de variables

Antes de decidir usar PCA, conviene compararlo con la alternativa más simple:
seleccionar directamente las N variables más relevantes.


In [13]:
tabla = pd.DataFrame({
    'Criterio': [
        'Interpretabilidad del resultado',
        'Reducción de dimensionalidad',
        'Manejo de correlación entre variables',
        'Cuándo usarlo',
        'Cuándo evitarlo',
        'Coste computacional',
    ],
    'PCA': [
        'Baja - variables sintéticas sin nombre natural',
        'Máxima - comprime toda la información disponible',
        'Excelente - elimina redundancia matemáticamente',
        'Dataset pequeño, variables muy correlacionadas, visualización 2D',
        'Cuando el negocio exige explicar cada variable',
        'Bajo - operación matricial O(n·p²)',
    ],
    'Selección de variables': [
        'Alta - conserva las variables originales',
        'Moderada - elimina variables, no las combina',
        'Parcial - puede dejar correlaciones si no se analiza',
        'Cuando la interpretabilidad es requisito de negocio',
        'Con cientos de variables muy correlacionadas',
        'Variable - depende del método de selección',
    ],
})

print(tabla.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
tbl = ax.table(
    cellText=tabla.values,
    colLabels=tabla.columns,
    cellLoc='left', loc='center',
    colWidths=[0.28, 0.36, 0.36]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 2.2)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
ax.set_title('PCA vs Selección de variables', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(IMG / 'B02D_fig05_tabla.png', dpi=120, bbox_inches='tight')
plt.close()
print("\nGuardado: B02D_fig05_tabla.png")


                             Criterio                                                              PCA                               Selección de variables
      Interpretabilidad del resultado                   Baja -  variables sintéticas sin nombre natural             Alta -  conserva las variables originales
         Reducción de dimensionalidad                 Máxima -  comprime toda la información disponible         Moderada -  elimina variables, no las combina
Manejo de correlación entre variables                  Excelente -  elimina redundancia matemáticamente Parcial -  puede dejar correlaciones si no se analiza
                        Cuándo usarlo Dataset pequeño, variables muy correlacionadas, visualización 2D  Cuando la interpretabilidad es requisito de negocio
                      Cuándo evitarlo                   Cuando el negocio exige explicar cada variable         Con cientos de variables muy correlacionadas
                  Coste computacional                     

## 11 · Visualización interactiva: clientes en espacio PCA (Plotly)

In [14]:
df_plot = df.copy()
df_plot['PC1'] = X_2d[:, 0]
df_plot['PC2'] = X_2d[:, 1]

fig_px = px.scatter(
    df_plot, x='PC1', y='PC2', color='segmento',
    hover_data=['login_frecuencia', 'modulos_activos', 'nps', 'tickets_mes'],
    color_discrete_map={'intensivo': '#2196F3', 'moderado': '#FF9800', 'bajo': '#F44336'},
    title=f'Clientes en espacio PCA-2D ({(v1+v2)*100:.1f}% varianza captada)',
    labels={'PC1': f'PC1 ({v1*100:.1f}% var)', 'PC2': f'PC2 ({v2*100:.1f}% var)'},
    opacity=0.75,
    template='plotly_white',
)

try:
    fig_px.write_image(str(IMG / 'B02D_fig06.png'), width=800, height=500)
    print("Guardado: B02D_fig06.png")
except Exception:
    fig_px.write_html(str(IMG / 'B02D_fig06.html'))
    print("Guardado: B02D_fig06.html (kaleido no disponible)")


Guardado: B02D_fig06.html (kaleido no disponible)


## 12 · Ejercicio técnico

**Objetivo:** comparar el error de un modelo de regresión entrenado con
todas las variables originales frente al mismo modelo entrenado con los
k componentes principales que retienen ≥85% de la varianza.

**Variable objetivo sintética:** `nps` (puntuación de satisfacción del cliente).

```python
# TODO: completa el código para entrenar y comparar los dos modelos
```


In [15]:
# ── Setup del ejercicio ──────────────────────────────────────────────────────
y = df['nps'].values.astype(float)
X_orig = df[FEATURES].values.astype(float)

X_train_o, X_test_o, y_train, y_test = train_test_split(
    X_orig, y, test_size=0.25, random_state=42)

# Modelo A: todas las variables originales (sin PCA)
scaler_a = StandardScaler()
X_train_a = scaler_a.fit_transform(X_train_o)
X_test_a  = scaler_a.transform(X_test_o)

model_a = Ridge(alpha=1.0)
model_a.fit(X_train_a, y_train)
mse_a = mean_squared_error(y_test, model_a.predict(X_test_a))

# ── TODO ─────────────────────────────────────────────────────────────────────
# Entrena el modelo B usando PCA con k componentes (retener ≥85% varianza)
# Pista: usa Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=k_85)), ('model', Ridge())])
# mse_b = ?

# ── Solución comentada ───────────────────────────────────────────────────────
pipe_b = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=k_85)),
    ('model',  Ridge(alpha=1.0)),
])
pipe_b.fit(X_train_o, y_train)
mse_b = mean_squared_error(y_test, pipe_b.predict(X_test_o))

print("Comparación de modelos - MSE en test")
print("-" * 40)
print(f"  Modelo A (8 variables originales)  MSE = {mse_a:.2f}")
print(f"  Modelo B ({k_85} componentes PCA)       MSE = {mse_b:.2f}")
print(f"  Diferencia absoluta: {abs(mse_a - mse_b):.2f}")
print()
if mse_b <= mse_a * 1.10:
    print("✓ El modelo con PCA obtiene un MSE similar con menos variables.")
    print("  Reducción de dimensionalidad: de 8 a", k_85, "variables.")
else:
    print("⚠ En este dataset el modelo completo es mejor - lo que también es")
    print("  un resultado válido: PCA no siempre ayuda con datos de alta calidad.")


Comparación de modelos -  MSE en test
----------------------------------------
  Modelo A (8 variables originales)  MSE = 0.25
  Modelo B (3 componentes PCA)       MSE = 86.26
  Diferencia absoluta: 86.01

⚠ En este dataset el modelo completo es mejor -  lo que también es
  un resultado válido: PCA no siempre ayuda con datos de alta calidad.


## 13 · Ejercicio de decisión

**Caso:** el equipo de ventas quiere un modelo que les diga qué clientes
tienen mayor riesgo de no renovar el contrato. El modelo usa 40 variables
extraídas del ERP. El director comercial ha pedido que el modelo sea
**explicable**: cuando el sistema muestra "riesgo alto", el vendedor debe
poder decir exactamente *qué* está disparando esa alerta (por ejemplo:
"llevan 45 días sin hacer login y han abierto 6 tickets este mes").

El equipo de datos propone usar PCA para comprimir las 40 variables en
8 componentes antes de entrenar el modelo.

---

**¿Aplicarías PCA en este caso? ¿Por qué sí o por qué no?**

*(Escribe tu razonamiento en la celda siguiente. No hay código necesario.)*


**Tu respuesta:**

*(escribe aquí)*

---

<details>
<summary>Criterios de evaluación (desplegar tras reflexionar)</summary>

**Respuesta sólida menciona:**
- Que PCA destruye la interpretabilidad de las variables originales: los
  componentes no tienen nombre de negocio, no se pueden explicar al vendedor.
- Que el requisito de "explicabilidad" es un criterio de diseño que debe
  respetarse antes de elegir la técnica.
- Que la alternativa correcta en este caso sería selección de variables
  (elegir las 8-10 más relevantes conservando sus nombres originales) o
  usar un modelo con importancia de features (Random Forest, XGBoost + SHAP).

**Señal de comprensión:** el alumno entiende que la elección técnica depende
del requisito de negocio, no solo del rendimiento estadístico del modelo.

**Error frecuente:** pensar que PCA es siempre mejor porque reduce el ruido.
PCA reduce dimensiones pero no siempre mejora el modelo, y tiene un coste
comunicativo real que puede hacer inutilizable el resultado en producción.

</details>


## 14 · Assets generados

| Archivo | Descripción |
|---|---|
| `data/metricas_uso_erp.csv` | 200 clientes, 10 features de uso de ERP + segmento |
| `images/B02D_fig01.png` | Scree plot: varianza por componente + varianza acumulada |
| `images/B02D_fig02.png` | Scatter 2D: clientes en espacio PC1 vs PC2 |
| `images/B02D_fig03.png` | Heatmap de loadings: contribución de variables a componentes |
| `images/B02D_fig04.png` | Varianza acumulada vs número de componentes k |
| `images/B02D_fig05_tabla.png` | Tabla comparativa: PCA vs selección de variables |
| `images/B02D_fig06.png/.html` | Scatter interactivo Plotly |


In [16]:
# Verificación final de assets
import os
assets = [
    DATOS / 'metricas_uso_erp.csv',
    IMG   / 'B02D_fig01.png',
    IMG   / 'B02D_fig02.png',
    IMG   / 'B02D_fig03.png',
    IMG   / 'B02D_fig04.png',
    IMG   / 'B02D_fig05_tabla.png',
]
print("Assets generados:")
for a in assets:
    existe = "✓" if a.exists() else "✗ FALTA"
    print(f"  {existe}  {a}")


Assets generados:
  ✓  datos\metricas_uso_erp.csv
  ✓  images\B02D_fig01.png
  ✓  images\B02D_fig02.png
  ✓  images\B02D_fig03.png
  ✓  images\B02D_fig04.png
  ✓  images\B02D_fig05_tabla.png
